In [ ]:
# =========================
# RQ4: Feature Importance Analysis
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

# -------------------------
# 1. Load Data
# -------------------------
df = pd.read_csv(
    "/kaggle/input/datasets/sharmajicoder/gaming-and-mental-health/gaming_mental_health_10M_40features.csv"
)

df = df.dropna()
df = df.sample(n=20000, random_state=42)

TARGET = df.columns[-1]

# -------------------------
# 2. Features and Target
# -------------------------
le = LabelEncoder()
y_raw = le.fit_transform(df[TARGET])

median_value = np.median(y_raw)
y = (y_raw > median_value).astype(int)

print("Class distribution:")
print(pd.Series(y).value_counts())

# -------------------------
# 3. Train/Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------------
# 4. Train Model
# -------------------------
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# -------------------------
# 5. Model Performance Check
# -------------------------
y_pred = model.predict(X_test)

print("Model Accuracy:", accuracy_score(y_test, y_pred))
print("Model F1-score:", f1_score(y_test, y_pred, zero_division=0))

# -------------------------
# 6. Permutation Importance
# -------------------------
perm = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": perm.importances_mean,
    "Std": perm.importances_std
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

importance_df.to_csv("RQ4_feature_importance_table.csv", index=False)

print("\n=== Top 10 Important Features ===")
print(importance_df.head(10))

# -------------------------
# 7. Plot Top 10 Features (Clean Version)
# -------------------------
top10 = importance_df.head(10).sort_values("Importance", ascending=True)

plt.figure(figsize=(9, 6))

# Use different colors
colors = plt.cm.tab10(np.linspace(0, 1, len(top10)))

plt.barh(
    top10["Feature"],
    top10["Importance"],
    color=colors
)

plt.title(
    "RQ4: Top Features Influencing Mental Health Risk",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Permutation Importance", fontsize=12)
plt.ylabel("Feature", fontsize=12)

# Add value labels (optional but nice)
for i, v in enumerate(top10["Importance"]):
    plt.text(v + 0.0005, i, f"{v:.3f}", va='center')

plt.grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("RQ4_feature_importance.pdf", bbox_inches="tight")
plt.show()